This notebook is used to synthesize the trained models with HLS4MLs backend Vitis Unified. Some synthesis include bitfile-generation, and varies with strategy.

In [15]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
from sklearn.metrics import accuracy_score

# Load Vitis into path
os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

In [16]:
model_to_test = 'jettag-hgq2-timings'

model_configs = [
    # AdaptiveHP-models
    {
        "description": "AdaptiveHP acc=0.7117 ebops=303 Distributed Arithmetic",
        "model_revision": "Training_AdaptiveHP",
        "keras_model_path": "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7117_ebops=303.keras",
        "hls4ml_strategy": "DA",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "acc=0.7117_ebops=303_VU_DA_bitfile",
    },
    #{
    #    "description": "AdaptiveHP acc=0.7117 ebops=303 latency",
    #    "model_revision": "Training_AdaptiveHP",
    #    "keras_model_path": "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7117_ebops=303.keras",
    #    "hls4ml_strategy": "latency",
    #    "hls4ml_generate_bitfile": True,
    #    "hls4ml_revision": "acc=0.7117_ebops=303_VU_latency_bitfile",
    #},

    {
        "description": "AdaptiveHP acc=0.7426 ebops=1001 Distributed Arithmetic",
        "model_revision": "Training_AdaptiveHP",
        "keras_model_path": "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7426_ebops=1001.keras",
        "hls4ml_strategy": "DA",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "acc=0.7426_ebops=1001_VU_DA_bitfile",
    },
    {
        "description": "AdaptiveHP acc=0.7512 ebops=2895 Distributed Arithmetic",
        "model_revision": "Training_AdaptiveHP",
        "keras_model_path": "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7512_ebops=2895.keras",
        "hls4ml_strategy": "DA",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "acc=0.7512_ebops=2895_VU_DA_bitfile",
    },
    # FixedHP-models
    {
        "description": "FixedHP acc=0.7590 ebops=11634 Distributed Arithmetic",
        "model_revision": "Training_FixedHP",
        "keras_model_path": "jettag-hgq2/Training_FixedHP/model_Training_FixedHP_acc=0.7590_ebops=11634.keras",
        "hls4ml_strategy": "DA",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "acc=0.7590_ebops=11634_VU_DA_bitfile",
    },

    {
        "description": "FixedHP acc=0.7659 ebops=21624 Distributed Arithmetic",
        "model_revision": "Training_FixedHP",
        "keras_model_path": "jettag-hgq2/Training_FixedHP/model_Training_FixedHP_acc=0.7659_ebops=21624.keras",
        "hls4ml_strategy": "DA",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "acc=0.7659_ebops=21624_VU_DA_bitfile",
    },
    #{
    #    "description": "FixedHP acc=0.7659 ebops=21624 latency",
    #    "model_revision": "Training_FixedHP",
    #    "keras_model_path": "jettag-hgq2/Training_FixedHP/model_Training_FixedHP_acc=0.7659_ebops=21624.keras",
    #    "hls4ml_strategy": "latency",
    #    "hls4ml_generate_bitfile": True,
    #    "hls4ml_revision": "acc=0.7659_ebops=21624_VU_latency_bitfile",
    #},
]

In [17]:
# Load dataset which is preprocessed in another notebook
# Used for calibration (x) and simulation, which we do not do here
#X_train = np.load("Data/processed_data/X_train.npy")
#X_val = np.load("Data/processed_data/X_val.npy")
x_test = np.load('Data/x_test.npy') # used for calibrating
#y_train = np.load("Data/processed_data/y_train.npy")
#y_val = np.load("Data/processed_data/y_val.npy")
#y_test = np.load("Data/processed_data/y_test.npy")
x_test.dtype


dtype('float32')

In [18]:
import os

def prepare_directory(model_config):
    output_dir = os.path.join(
        os.path.dirname(os.path.abspath(model_config["keras_model_path"])),
        f"hls4ml_prj_{model_config['hls4ml_revision']}",
    )
    os.makedirs(output_dir, exist_ok=True)

    description = f"""
    Description of HLS4ML-project.

    {model_config['description']}

    - Bitfile: {model_config['hls4ml_generate_bitfile']}
    - Environment: devenv-vu-hgq+da (environment-HGQ+DA.yml)
    - Target Device: KV260 (xck26-sfvc784-2LV-c)
    - Dataset: HLS4ML LHC Jets
    - Vivado/Vitis: 2025.2
    - Model Architecture: {model_to_test}
    - Model Revision: {model_config['model_revision']}
    - HLS4ML Revision: {model_config['hls4ml_revision']}

    The model summary is in the parent-directory, `summary.txt`
    """
    with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
        f.write(description)

    return output_dir

In [ ]:
import time
import numpy as np

def test_model(keras_model_path):
    model = load_model(keras_model_path)

    # Timing
    runs = 10
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        model.predict(x_test, verbose=0)
        times.append(time.perf_counter() - start)

    print(times)

    return model

In [27]:
# Run through every model
for i, model_config in enumerate(model_configs):
    print(f"\nProcessing model {i+1}/{len(model_configs)}: {model_config['description']}")
    print(f"%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")

    hls_model = test_model(
        model_config["keras_model_path"]
    )


Processing model 1/5: AdaptiveHP acc=0.7117 ebops=303 Distributed Arithmetic
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
[1.9092614511027932, 0.6667320188134909, 0.6652787358034402, 0.7079866810236126, 0.7105177389457822, 0.6738513610325754, 0.6771678768564016, 0.6747167310677469, 0.7252490611281246, 0.7003484179731458]

Processing model 2/5: AdaptiveHP acc=0.7426 ebops=1001 Distributed Arithmetic
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
[2.9130353161599487, 0.6862454551737756, 0.7162085559684783, 0.7245421239640564, 0.861455992795527, 0.8367377279791981, 0.7545330270659178, 0.7237313769292086, 0.686714650830254, 0.6712969390209764]

Processing model 3/5: AdaptiveHP acc=0.7512 ebops=2895 Distributed Arithmetic
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
[1.9564137470442802, 0.6875371998175979, 0.6748664369806647, 0.6996922090183944, 0.6818928129505366, 0.7067536190152168, 0.7253335840068758, 0.7206601980142295, 0.6924323181156069, 1.2912182279396802]

Processing model 4/5: FixedHP acc=0.